# Basic example

### 4 hours of prices, using only basic battery info

In [1]:
hourly_prices = [3,6,2,5] #units £/MWh

In [2]:
max_charging_rate = 2 #units MW
max_discharging_rate = 2 #units MW
max_storage_volume = 4 #units MWh

In [3]:
charging_efficiency = 0.05 #Fraction of energy imported from grid that is lost prior to storage in the battery
discharging_efficiency = 0.05 #Fraction of energy exported from the battery that is lost prior to reaching the grid

### Aim: maxmise profit, can decide when to charge and discharge, can't decide what prices are

In [4]:
import pulp

In [5]:
model = pulp.LpProblem("battery_optimisation", pulp.LpMaximize)

In [6]:
n_hours = len(hourly_prices)
print(n_hours)

4


In [7]:
# Define variables at each time step, state of charge (soc) is useful to link charge and discharge
charge = pulp.LpVariable.dicts("charge", range(n_hours), 0, max_charging_rate)
discharge = pulp.LpVariable.dicts("discharge", range(n_hours), 0, max_discharging_rate)
soc = pulp.LpVariable.dicts("soc", range(n_hours), 0, max_storage_volume)

In [8]:
print(discharge[2])

discharge_2


In [9]:
# profit = sum( price * discharge * efficiency - price * charge )

model += pulp.lpSum(hourly_prices[t] * discharge[t] * (1 - discharging_efficiency) - hourly_prices[t] * charge[t] for t in range(n_hours))

In [10]:
# state of charge = state of charge at previous hour + charge * efficiency - discharge  --- this links charge and discharge together, essentially conservation of energy

initial_soc = 0.0

for t in range(n_hours):
    previous_soc = initial_soc if t == 0 else soc[t - 1]
    model += soc[t] == previous_soc + charge[t] * (1 - charging_efficiency) - discharge[t]

In [11]:
model.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/constancemahony/miniconda3/envs/pulp-env/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/55/qvzldl952577jnfzcs8thwqc0000gn/T/fdd781a570f84bc9b16c9919bbbdf570-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/55/qvzldl952577jnfzcs8thwqc0000gn/T/fdd781a570f84bc9b16c9919bbbdf570-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 33 RHS
At line 38 BOUNDS
At line 51 ENDATA
Problem MODEL has 4 rows, 12 columns and 15 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 3 (-1) rows, 9 (-3) columns and 11 (-4) elements
0  Obj -0 Dual inf 12.349997 (3)
4  Obj 9.855
Optimal - objective value 9.855
After Postsolve, objective 9.855, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 9.855 - 4 iterations time 0.002, Presolve 0.00
O

1

### Print output

In [12]:
print(f"Status: {pulp.LpStatus[model.status]}\n")
for t in range(n_hours):
    print(f"Hour {t}: price=£{hourly_prices[t]:3d}  "
          f"charge={charge[t].varValue:.2f}  "
          f"discharge={discharge[t].varValue:.2f}  "
          f"SoC={soc[t].varValue:.2f}")

profit = pulp.value(model.objective)
print(f"\nTotal profit: £{profit:.2f}")

Status: Optimal

Hour 0: price=£  3  charge=2.00  discharge=0.00  SoC=1.90
Hour 1: price=£  6  charge=0.00  discharge=1.90  SoC=0.00
Hour 2: price=£  2  charge=2.00  discharge=0.00  SoC=1.90
Hour 3: price=£  5  charge=0.00  discharge=1.90  SoC=0.00

Total profit: £9.85


### add schedule to dataframe

In [5]:
import pandas as pd

In [14]:
schedule = pd.DataFrame({"price": hourly_prices,
                         "charge": [charge[t].varValue for t in range(n_hours)],
                         "discharge": [discharge[t].varValue for t in range(n_hours)],
                         "soc": [soc[t].varValue for t in range(n_hours)]})

In [15]:
print(schedule)

   price    charge     discharge           soc
0      6  0.000000  1.000000e-12 -1.000000e-12
1      3  0.105263  0.000000e+00  1.000000e-01
2      2  2.000000  0.000000e+00  2.000000e+00
3      5  0.000000  2.000000e+00  0.000000e+00


# write optimiser function

In [15]:
def battery_optimiser(hourly_prices, max_charging_rate, max_discharging_rate, max_storage_volume, charging_efficiency, discharging_efficiency, initial_soc, solver=pulp.PULP_CBC_CMD(msg=0)):
    
    model = pulp.LpProblem("battery_optimisation", pulp.LpMaximize)
    
    n_hours = len(hourly_prices)
    charge = pulp.LpVariable.dicts("charge", range(n_hours), 0, max_charging_rate)
    discharge = pulp.LpVariable.dicts("discharge", range(n_hours), 0, max_discharging_rate)
    soc = pulp.LpVariable.dicts("soc", range(n_hours), 0, max_storage_volume)
   
    model += pulp.lpSum(hourly_prices[t] * discharge[t] * (1 - discharging_efficiency) - hourly_prices[t] * charge[t] for t in range(n_hours))
    
    for t in range(n_hours):
        previous_soc = initial_soc if t == 0 else soc[t - 1]
        model += soc[t] == previous_soc + charge[t] * (1 - charging_efficiency) - discharge[t]

    model.solve(solver)

    schedule = pd.DataFrame({"price": hourly_prices,
                             "charge": [charge[t].varValue for t in range(n_hours)],
                             "discharge": [discharge[t].varValue for t in range(n_hours)],
                             "soc": [soc[t].varValue for t in range(n_hours)]})
    
    profit = pulp.value(model.objective)

    return pulp.LpStatus[model.status], schedule, profit

In [16]:
battery_optimiser(hourly_prices, max_charging_rate, max_discharging_rate, max_storage_volume, charging_efficiency, discharging_efficiency, initial_soc)

('Optimal',
    price    charge     discharge           soc
 0      6  0.000000  1.000000e-12 -1.000000e-12
 1      3  0.105263  0.000000e+00  1.000000e-01
 2      2  2.000000  0.000000e+00  2.000000e+00
 3      5  0.000000  2.000000e+00  0.000000e+00,
 5.1842105200057)

# Now read in example data

In [37]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_excel("data/prices_example.xlsx", sheet_name="Hourly data", names=["date_time", "price"], index_col="date_time")

In [28]:
print(df.head())

                     price
date_time                 
2018-01-01 00:00:00  51.89
2018-01-01 01:00:00  55.49
2018-01-01 02:00:00  51.06
2018-01-01 03:00:00  44.76
2018-01-01 04:00:00  40.07


In [29]:
print(df.info)

<bound method DataFrame.info of                           price
date_time                      
2018-01-01 00:00:00.000  51.890
2018-01-01 01:00:00.000  55.490
2018-01-01 02:00:00.000  51.060
2018-01-01 03:00:00.000  44.760
2018-01-01 04:00:00.000  40.070
...                         ...
2020-12-31 18:59:59.994  72.415
2020-12-31 19:59:59.994  73.610
2020-12-31 20:59:59.994  69.735
2020-12-31 21:59:59.994  64.450
2020-12-31 22:59:59.994  66.235

[26304 rows x 1 columns]>


In [33]:
hourly_prices = df["price"].values

In [34]:
print(hourly_prices)

[51.89  55.49  51.06  ... 69.735 64.45  66.235]


### Create function to read in data

In [35]:
def load_prices_from_excel(filename, sheetname):
    df = pd.read_excel(filename, sheet_name=sheetname, names=["date_time", "price"], index_col="date_time")
    hourly_prices = df["price"].values
    return hourly_prices

In [36]:
hourly_prices_function_test = load_prices_from_excel("data/prices_example.xlsx", "Hourly data")

In [38]:
np.array_equal(hourly_prices, hourly_prices_function_test)

True

# Combine everything together

In [1]:
import pandas as pd
import pulp

In [2]:
def load_prices_from_excel(filename, sheetname):
    df = pd.read_excel(filename, sheet_name=sheetname, names=["date_time", "price"], index_col="date_time")
    hourly_prices = df["price"].values
    return hourly_prices

In [3]:
hourly_prices = load_prices_from_excel("data/prices_example.xlsx", "Hourly data")

In [4]:
print(hourly_prices)

[51.89  55.49  51.06  ... 69.735 64.45  66.235]


In [5]:
def battery_optimiser(hourly_prices, max_charging_rate, max_discharging_rate, max_storage_volume, charging_efficiency, discharging_efficiency, initial_soc, solver=pulp.PULP_CBC_CMD(msg=0)):
    
    model = pulp.LpProblem("battery_optimisation", pulp.LpMaximize)
    
    n_hours = len(hourly_prices)
    charge = pulp.LpVariable.dicts("charge", range(n_hours), 0, max_charging_rate)
    discharge = pulp.LpVariable.dicts("discharge", range(n_hours), 0, max_discharging_rate)
    soc = pulp.LpVariable.dicts("soc", range(n_hours), 0, max_storage_volume)
   
    model += pulp.lpSum(hourly_prices[t] * discharge[t] * (1 - discharging_efficiency) - hourly_prices[t] * charge[t] for t in range(n_hours))
    
    for t in range(n_hours):
        previous_soc = initial_soc if t == 0 else soc[t - 1]
        model += soc[t] == previous_soc + charge[t] * (1 - charging_efficiency) - discharge[t]

    model.solve(solver)

    schedule = pd.DataFrame({"price": hourly_prices,
                             "charge": [charge[t].varValue for t in range(n_hours)],
                             "discharge": [discharge[t].varValue for t in range(n_hours)],
                             "soc": [soc[t].varValue for t in range(n_hours)]})
    
    profit = pulp.value(model.objective)

    return pulp.LpStatus[model.status], schedule, profit

In [6]:
max_charging_rate = 2 #units MW
max_discharging_rate = 2 #units MW
max_storage_volume = 4 #units MWh
charging_efficiency = 0.05 #Fraction of energy imported from grid that is lost prior to storage in the battery
discharging_efficiency = 0.05 #Fraction of energy exported from the battery that is lost prior to reaching the grid
initial_soc = 0.0

In [7]:
battery_optimiser(hourly_prices, max_charging_rate, max_discharging_rate, max_storage_volume, charging_efficiency, discharging_efficiency, initial_soc)

('Optimal',
         price    charge  discharge  soc
 0      51.890  0.000000        0.0  0.0
 1      55.490  0.000000        0.0  0.0
 2      51.060  0.000000        0.0  0.0
 3      44.760  0.210526        0.0  0.2
 4      40.070  2.000000        0.0  2.1
 ...       ...       ...        ...  ...
 26299  72.415  0.000000        0.0  0.0
 26300  73.610  0.000000        0.0  0.0
 26301  69.735  0.000000        0.0  0.0
 26302  64.450  0.000000        0.0  0.0
 26303  66.235  0.000000        0.0  0.0
 
 [26304 rows x 4 columns],
 174052.80789141203)

# Tests

In [ ]:
status = pulp.LpStatus[model.status]
    if status != "Optimal":
        raise RuntimeError(f"Solver did not find an optimal solution. Status: {status}")


In [5]:
hourly_prices = [3,6,2,5] #units £/MWh

In [16]:
def test_basic_example_with_defaults():
    prices = [3,6,2,5]
    schedule, profit = battery_optimiser_new(prices)
    assert profit == 9.854999999999999

In [17]:
test_basic_example_with_defaults()

### Improved version of battery optimiser

In [14]:
def battery_optimiser_new(hourly_prices, max_charging_rate=2, max_discharging_rate=2, max_storage_volume=4, charging_efficiency=0.05, discharging_efficiency=0.05, initial_soc=0.0, solver=pulp.PULP_CBC_CMD(msg=0)):
    
    model = pulp.LpProblem("battery_optimisation", pulp.LpMaximize)
    
    n_hours = len(hourly_prices)
    charge = pulp.LpVariable.dicts("charge", range(n_hours), 0, max_charging_rate)
    discharge = pulp.LpVariable.dicts("discharge", range(n_hours), 0, max_discharging_rate)
    soc = pulp.LpVariable.dicts("soc", range(n_hours), 0, max_storage_volume)
   
    model += pulp.lpSum(hourly_prices[t] * discharge[t] * (1 - discharging_efficiency) - hourly_prices[t] * charge[t] for t in range(n_hours))
    
    for t in range(n_hours):
        previous_soc = initial_soc if t == 0 else soc[t - 1]
        model += soc[t] == previous_soc + charge[t] * (1 - charging_efficiency) - discharge[t]

    model.solve(solver)

    status = pulp.LpStatus[model.status]
    #status = "Infeasible"
    if status != "Optimal":
        raise RuntimeError(f"Solver didn't find optimal solution. Status:{status}")

    schedule = pd.DataFrame({"price": hourly_prices,
                             "charge": [charge[t].varValue for t in range(n_hours)],
                             "discharge": [discharge[t].varValue for t in range(n_hours)],
                             "soc": [soc[t].varValue for t in range(n_hours)]})
    
    profit = pulp.value(model.objective)

    return schedule, profit

In [10]:
battery_optimiser_new(hourly_prices, max_charging_rate, max_discharging_rate, max_storage_volume, charging_efficiency, discharging_efficiency, initial_soc)

(   price  charge  discharge  soc
 0      3     2.0        0.0  1.9
 1      6     0.0        1.9  0.0
 2      2     2.0        0.0  1.9
 3      5     0.0        1.9  0.0,
 9.854999999999999)

In [11]:
battery_optimiser_new(hourly_prices)

RuntimeError: Solver didn't find optimal solution. Status:Infeasible

In [14]:
battery_optimiser_new(load_prices_from_excel("data/prices_example.xlsx", "Hourly data"))

(        price    charge  discharge  soc
 0      51.890  0.000000        0.0  0.0
 1      55.490  0.000000        0.0  0.0
 2      51.060  0.000000        0.0  0.0
 3      44.760  0.210526        0.0  0.2
 4      40.070  2.000000        0.0  2.1
 ...       ...       ...        ...  ...
 26299  72.415  0.000000        0.0  0.0
 26300  73.610  0.000000        0.0  0.0
 26301  69.735  0.000000        0.0  0.0
 26302  64.450  0.000000        0.0  0.0
 26303  66.235  0.000000        0.0  0.0
 
 [26304 rows x 4 columns],
 174052.80789141203)